# Step 4b: Claim detector v2 (health talk + auto-captions)
Step 4 scored well on its own test split but failed on what we actually need:
- it caught only 2 of 5 health claims in a demo transcript (casual causal claims like "X cures Y" looked like opinions), and
- it flagged **0** claims on unpunctuated lowercase text, which is what YouTube auto-captions look like.

**What changes in training**
1. **Health claims as positives**: HealthVer + SciFact train claims (already on your Drive in `unified_v2.parquet`).
2. **Opinion sentences as negatives**: CheckThat! 2025 subjectivity data (English, subjective sentences from news).
3. **Auto-caption robustness**: 20% of training sentences also appear lowercased with punctuation stripped, plus 3,000 multi-sentence "windows" (label = contains a claim).
4. **A small amount of synthetic health-style sentences** (templates over made-up topics): casual causal claims as positives, opinion/filler as negatives. Disclose this in the pitch.

**Evaluation.** Everything is held out from training: CT24 dev-test, health claims (HealthVer val/test, SciFact val), CT25 subjective dev-test sentences, unpunctuated windows, and synthetic sentences with unseen phrasing. v1 and v2 are scored side by side.

**Runtime → T4 GPU.** Approve the Drive pop-up. About 15 minutes.

In [ ]:
%pip install -q "datasets==3.6.0" "transformers>=4.46,<5" accelerate sentencepiece protobuf scikit-learn pandas pyarrow

In [ ]:
import os, re, json, random, numpy as np, pandas as pd, torch
from datasets import load_dataset, Dataset
from sklearn.metrics import f1_score, precision_score, recall_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer,
                          DataCollatorWithPadding, set_seed)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/veritas"
except ImportError:                       # not on Colab (local test only)
    DRIVE = "veritas_local"
os.makedirs(DRIVE, exist_ok=True)
SAVE_V1, SAVE_V2 = f"{DRIVE}/claim_detector", f"{DRIVE}/claim_detector_v2"
CKPT = "/content/ckpt_claims2" if os.path.isdir("/content") else "ckpt_claims2"
BASE = "microsoft/deberta-v3-base"
MAXLEN, EPOCHS, LR, BS = 128, 3, 2e-5, 32
N_DEGRADED, N_WINDOWS, N_TPL_POS = 0.20, 3000, 600
DEV = "cuda" if torch.cuda.is_available() else "cpu"
set_seed(42); rng = random.Random(42)
print("device:", DEV)

## 1. Data
Every source ends up as `(text, label)` with label 1 = checkable claim, 0 = not.

In [ ]:
clean = lambda s: re.sub(r"\s+", " ", str(s)).strip()

# CheckThat! 2024 (political debates). Note: the 'test' split here is the DEV-TEST set (318), not the hidden official test.
ds = load_dataset("iai-group/clef2024_checkthat_task1_en")
ct = {k: pd.DataFrame(dict(text=ds[s]["Text"], label=[int(x == "Yes") for x in ds[s]["class_label"]]))
      for k, s in [("train", "train"), ("val", "validation"), ("test", "test")]}
for d in ct.values(): d["text"] = d.text.map(clean)

# Health claims: HealthVer + SciFact from the unified table, using each dataset's own splits
upath = next(p for p in [f"{DRIVE}/unified_v2.parquet", "/tmp/unified_v2.parquet", "unified_v2.parquet"] if os.path.exists(p))
U = pd.read_parquet(upath); U["claim"] = U.claim.map(clean)
hp = lambda src, sp: sorted(set(U[(U.source.isin(src)) & (U.split == sp)].claim))
h_train = hp(["healthver", "scifact"], "train")
h_val   = hp(["healthver"], "validation")
h_eval  = sorted(set(hp(["healthver"], "test")) | set(hp(["scifact"], "validation")))
seen = set(h_train); h_val = [t for t in h_val if t not in seen]; seen |= set(h_val); h_eval = [t for t in h_eval if t not in seen]

# Opinion sentences (CheckThat! 2025 subjectivity, English): SUBJ = negatives
sj = load_dataset("AIWizards/clef2025_checkthat_task1_subjectivity", "english")
subj = lambda sp: [clean(s) for s, l in zip(sj[sp]["sentence"], sj[sp]["label"]) if l == "SUBJ"]
s_train, s_val, s_eval = subj("train"), subj("dev"), subj("dev_test")   # the official test split has no labels

# Synthetic health-style sentences (made-up topics). Train and eval use different topics, conditions and phrasings.
cap = lambda s: s[0].upper() + s[1:]
T_TR = ["sauna sessions", "yoga", "meditation", "green tea", "turmeric", "a keto diet", "protein shakes", "red light therapy",
        "collagen powder", "walking after dinner", "journaling", "probiotics", "apple cider vinegar", "creatine",
        "morning sunlight", "daily stretching", "fish oil"]
T_EV = ["ginger tea", "pilates", "breathwork", "bone broth"]
C_TR = ["heart disease", "type 2 diabetes", "high blood pressure", "depression", "migraines", "arthritis", "influenza", "dementia", "back pain", "acne"]
C_EV = ["osteoporosis", "asthma", "anxiety", "gout"]
POS_TR = ["{t} can cure {c}.", "Studies show that people who use {t} lower their risk of {c}.",
          "If you try {t} every day, you will never get {c}.", "Experts agree that {t} can prevent {c}.",
          "Scientists have proven that {t} can reverse {c}.", "My doctor told me {t} would help with {c}.",
          "I tried {t} for a month and it got rid of my {c}.", "Researchers found that {t} makes {c} much worse."]
POS_EV = ["There is strong evidence that {t} can treat {c}.", "{t} can double your chance of developing {c}."]
NEG_TR = ["I really love {t}.", "Honestly, {t} is my favorite part of the whole routine.", "In my opinion, {t} is totally worth trying.",
          "I personally feel so much better when I do {t}.", "Today we're going to talk about {t}.",
          "Have you ever wondered whether {t} is worth it?", "Let me know in the comments what you think about {t}.",
          "What has your experience with {t} been like?", "Welcome back, in this video I'll share my thoughts on {t}.",
          "I think {t} is a bit overrated.", "Let's dive right into {t}."]
NEG_EV = ["Don't forget to subscribe if you enjoy videos about {t}.", "I'm so excited to finally try {t}.", "Honestly I can't decide how I feel about {t}."]
gen = lambda tm, tp, cd=(None,): [cap(x.format(t=t, c=c)) for x in tm for t in tp for c in cd]
syn_pos = rng.sample(gen(POS_TR, T_TR, C_TR), N_TPL_POS); syn_neg = gen(NEG_TR, T_TR)
syn_eval_pos, syn_eval_neg = gen(POS_EV, T_EV, C_EV), gen(NEG_EV, T_EV)

rows = [("CheckThat! 2024 debates", "train/val/dev-test", len(ct["train"]), len(ct["val"]), len(ct["test"])),
        ("Health claims (+)", "train/val/eval", len(h_train), len(h_val), len(h_eval)),
        ("Subjective sentences (-)", "train/val/eval", len(s_train), len(s_val), len(s_eval)),
        ("Synthetic (+)/(-)", "train/-/eval", f"{len(syn_pos)}/{len(syn_neg)}", "-", f"{len(syn_eval_pos)}/{len(syn_eval_neg)}")]
print(pd.DataFrame(rows, columns=["source", "splits", "train", "val", "eval"]).to_string(index=False))
print("\nexample synthetic +:", syn_pos[0], "|", syn_pos[1]); print("example synthetic -:", syn_neg[0], "|", syn_neg[1])

In [ ]:
PUNCT = re.compile(r"[.,!?;:\"“”]")
degrade = lambda s: re.sub(r"\s+", " ", PUNCT.sub("", s.lower())).strip()      # auto-caption style
def windows(pairs, n, seed, degrade_p=1.0):
    r = random.Random(seed); out = []
    for _ in range(n):
        k = r.choice([2, 3]); pick = r.sample(pairs, k)
        t = " ".join(p[0] for p in pick); t = degrade(t) if r.random() < degrade_p else t
        out.append((t, max(p[1] for p in pick)))
    return out

base = ([(t, l) for t, l in zip(ct["train"].text, ct["train"].label)] + [(t, 1) for t in h_train] +
        [(t, 0) for t in s_train] + [(t, 1) for t in syn_pos] + [(t, 0) for t in syn_neg])
degraded = [(degrade(t), l) for t, l in rng.sample(base, int(N_DEGRADED * len(base)))]
wins = windows(base, N_WINDOWS, seed=1, degrade_p=0.7)
TRAIN = pd.DataFrame(base + degraded + wins, columns=["text", "label"]).sample(frac=1, random_state=42).reset_index(drop=True)
print("training rows:", len(TRAIN), "| claim rate:", round(TRAIN.label.mean(), 3))

VAL = pd.DataFrame(list(zip(ct["val"].text, ct["val"].label)) + [(t, 1) for t in h_val] + [(t, 0) for t in s_val], columns=["text", "label"])
print("validation rows:", len(VAL), "| claim rate:", round(VAL.label.mean(), 3))
# held-out evaluation sets
EV = {"CT24 dev-test (318)": (ct["test"].text.tolist(), ct["test"].label.tolist()),
      "Health claims, all positive": (h_eval, [1] * len(h_eval)),
      "News opinions, all negative": (s_eval, [0] * len(s_eval)),
      "Unpunctuated windows (CT24)": tuple(map(list, zip(*windows(list(zip(ct["test"].text, ct["test"].label)), 300, seed=7)))),
      "Synthetic, unseen phrasing": (syn_eval_pos + syn_eval_neg, [1] * len(syn_eval_pos) + [0] * len(syn_eval_neg))}
print({k: len(v[0]) for k, v in EV.items()})

## 2. Train
Same recipe as step 4. The best epoch is chosen by F1 on the combined validation set (CT24 val + health claims + opinion sentences).

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE)
enc = lambda d: Dataset.from_dict(dict(text=d.text.tolist(), label=d.label.tolist())).map(
    lambda b: tok(b["text"], truncation=True, max_length=MAXLEN), batched=True, remove_columns=["text"])
dtr, dva = enc(TRAIN), enc(VAL)
metrics = lambda p: dict(f1=f1_score(p.label_ids, p.predictions.argmax(-1)),
                         precision=precision_score(p.label_ids, p.predictions.argmax(-1)),
                         recall=recall_score(p.label_ids, p.predictions.argmax(-1)))
model = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=2)
args = TrainingArguments(output_dir=CKPT, num_train_epochs=EPOCHS, learning_rate=LR, per_device_train_batch_size=BS,
                         per_device_eval_batch_size=64, warmup_ratio=0.1, weight_decay=0.01, eval_strategy="epoch",
                         save_strategy="epoch", save_total_limit=1, load_best_model_at_end=True, metric_for_best_model="f1",
                         fp16=torch.cuda.is_available(), report_to="none", seed=42)
trainer = Trainer(model=model, args=args, train_dataset=dtr, eval_dataset=dva,
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
trainer.train()

## 3. Threshold on validation, then v1 vs v2 on held-out sets
The cut-off is chosen on the combined **validation** set only. v1 uses its own saved cut-off (0.6). Positive-only and negative-only sets report **recall** and **false-alarm rate** instead of F1.

In [ ]:
@torch.no_grad()
def probs_of(texts, m, tk, bs=64):
    m.eval(); out = []
    for i in range(0, len(texts), bs):
        b = tk(texts[i:i + bs], truncation=True, max_length=MAXLEN, padding=True, return_tensors="pt").to(m.device)
        out.append(torch.softmax(m(**b).logits.float(), -1)[:, 1].cpu().numpy())
    return np.concatenate(out) if out else np.array([])

pv = probs_of(VAL.text.tolist(), trainer.model, tok); yv = VAL.label.values
grid = np.round(np.arange(0.1, 0.91, 0.05), 2); f1s = [f1_score(yv, pv >= t) for t in grid]
THR = float(grid[int(np.argmax(f1s))])
print("validation F1 by threshold:", dict(zip(grid.tolist(), np.round(f1s, 3).tolist())))
print("chosen threshold (validation only):", THR)

def report(m, tk, thr):
    res = {}
    for name, (texts, y) in EV.items():
        if not texts: continue
        p = probs_of(texts, m, tk) >= thr; y = np.array(y)
        if y.min() == y.max() == 1:   res[name] = f"recall {p.mean():.3f}"
        elif y.min() == y.max() == 0: res[name] = f"false-alarm rate {p.mean():.3f}"
        else:                         res[name] = f"F1 {f1_score(y, p):.3f} (P {precision_score(y, p):.2f} R {recall_score(y, p):.2f})"
    return res
table = {"v2 (this notebook)": report(trainer.model, tok, THR)}
try:
    t1 = AutoTokenizer.from_pretrained(SAVE_V1); m1 = AutoModelForSequenceClassification.from_pretrained(SAVE_V1).to(DEV)
    thr1 = json.load(open(f"{SAVE_V1}/extract_config.json"))["threshold"]
    table = {f"v1 (step 4, thr {thr1})": report(m1, t1, thr1), **table}
except Exception as e:
    print("v1 not found, showing v2 only:", repr(e)[:100])
print(); print(pd.DataFrame(table).T.to_string())

**How to read this.** Compare v1 and v2 row by row. Health-claim recall and unpunctuated-window F1 should rise a lot. CT24 dev-test F1 may dip a little, which is acceptable. Papers report about 0.90 on this dev-test set for RoBERTa; the hidden official test was harder: teams ranked 7th-12th scored 0.71-0.76 F1 there.

## 4. Save

In [ ]:
os.makedirs(SAVE_V2, exist_ok=True)
trainer.save_model(SAVE_V2); tok.save_pretrained(SAVE_V2)
json.dump(dict(base=BASE, threshold=THR, max_len=MAXLEN, version=2), open(f"{SAVE_V2}/extract_config.json", "w"))
print(sorted(os.listdir(SAVE_V2)))

## 5. Transcript → claims
Same splitter as step 4: strip timestamps, split on `. ! ?`, cut long unpunctuated runs into 25-word windows.

In [ ]:
def split_transcript(text, max_words=40, window=25):
    text = re.sub(r"\[?\(?\b\d{1,2}:\d{2}(?::\d{2})?\b\]?\)?", " ", text)
    text = re.sub(r"\s+", " ", text).strip(); out = []
    for s in re.split(r"(?<=[.!?])\s+", text):
        w = s.split()
        if len(w) <= max_words:
            if len(w) >= 4: out.append(s)
        else:
            out += [" ".join(w[i:i + window]) for i in range(0, len(w), window) if len(w[i:i + window]) >= 4]
    return out

def extract_claims(transcript, thr=None):
    thr = THR if thr is None else thr
    sents = split_transcript(transcript); p = probs_of(sents, trainer.model, tok)
    return [dict(sentence=s, p_claim=float(x), is_claim=bool(x >= thr)) for s, x in zip(sents, p)]

DEMO1 = '''[0:01] Hey everyone, welcome back to the channel. Today I want to talk about sleep and supplements.
[0:12] So first, I take magnesium glycinate every night and honestly it cures my insomnia. Studies show that vitamin D
supplementation reduces the risk of respiratory infections. I personally think cold plunges are amazing.
[0:40] Masks do not reduce the spread of COVID-19, that's just a fact. Intermittent fasting is the best diet in my opinion.
Drinking lemon water every morning cures cancer. Make sure you like and subscribe. The Eiffel Tower is 330 meters tall.
Statins cause more harm than good for most people. Anyway, let me know what you think in the comments.'''
DEMO2 = '''Okay so, um, last month my doctor told me my cholesterol was high. A lot of people say eggs raise your cholesterol
but the research really doesn't back that up. I swapped sugar for honey and my energy is way better. Drinking eight glasses
of water a day is necessary for good health. Seed oils are poison, they cause inflammation throughout the body.
Honestly this smoothie recipe is so good. Sunscreen causes more cancer than it prevents. If you walk ten thousand steps
a day you will live longer. Alright, that's it for today, thanks for watching.'''
pd.set_option("display.max_colwidth", 100)
for name, T in [("DEMO 1 (invented; its phrasing overlaps the synthetic templates, so not a clean test)", DEMO1),
                ("DEMO 2 (invented; different topics and phrasing from the templates)", DEMO2)]:
    r = pd.DataFrame(extract_claims(T)); print("\n" + name)
    print(r.assign(p_claim=r.p_claim.round(2)).to_string(index=False))
    flat = " ".join(T.split()).lower().replace(".", "").replace(",", "")
    print(f"  unpunctuated version: {len(split_transcript(flat))} chunks, {sum(x['is_claim'] for x in extract_claims(flat))} flagged")

## What to check
1. **The v1 vs v2 table.** Health-claim recall up (v1 was weak on casual health talk), unpunctuated-window F1 up, news-opinion false-alarm rate not much higher, CT24 dev-test F1 within a few points of v1.
2. **DEMO 2.** The clean probe. The claims (eggs/cholesterol, eight glasses of water, seed oils, sunscreen, ten thousand steps) should score high. Personal anecdotes ("my doctor told me", "swapped sugar for honey"), the smoothie remark and the sign-off should score low.
3. **Unpunctuated counts** should now be above 0 and in the same range as the punctuated version.
4. **The real test is still real transcripts.** All of this is checked on text I invented or trained around. Paste 2-3 real health-video transcripts and we label the claims by hand.